# Prefect workflow for running the s3l0 eopf processor with the rs-dpr-service

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-652

See the associated:

  * Python module: [s3l0_demo_processor.py](./s3l0_demo_processor.py)
  * YAML file: [s3l0_demo_processor.yaml](./s3l0_demo_processor.yaml)

## 1. Initialisation

In [1]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server.processing.svc.cluster.local:4200/api
Prefect dashboard public URL: https://processing.dev-rspy-ovh.esa-copernicus.eu/dashboard


In [2]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
from resources.prefect_utils import *
USE_DPR_MOCKUP = True
if os.getenv("RSPY_LOCAL_MODE") == "1" and USE_DPR_MOCKUP:
    os.environ["DASK_GATEWAY_EOPF_ADDRESS"] = os.environ["DASK_GATEWAY_EOPF_MOCKUP_ADDRESS"]
    os.environ["DASK_GATEWAY_EOPF_PUBLIC"] = os.environ["DASK_GATEWAY_EOPF_MOCKUP_PUBLIC"]

init_demo()
init_dask_cluster_eopf(scale=2, use_mockup = USE_DPR_MOCKUP)
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  
from resources.prefect_utils import * 

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)
display(dask_cluster_staging)

DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.1"


Auxip service: https://dev-rspy-ovh.esa-copernicus.eu/auxip
CADIP service: https://dev-rspy-ovh.esa-copernicus.eu/cadip
Catalog service: https://dev-rspy-ovh.esa-copernicus.eu
Staging service: https://dev-rspy-ovh.esa-copernicus.eu
Connecting to dask gateway for 'dask-eopf-mockup': http://traefik-dask-gateway.dask-gateway.svc.cluster.local ...
image = dask-gateway.813ff94f7e8c4d3da11582762e08f13c
image = dask-gateway.21911191d10b4587b83ed82da1e8006d
Get existing dask cluster: 'dask-gateway.21911191d10b4587b83ed82da1e8006d'
Dask dashboard for 'dask-eopf-mockup': https://dash-dask.dev-rspy-ovh.esa-copernicus.eu/clusters/dask-gateway.21911191d10b4587b83ed82da1e8006d/status
Dask workers for 'dask-eopf-mockup' are up: 2/2
Connecting to dask gateway for 'dask-staging': http://traefik-dask-gateway.dask-gateway.svc.cluster.local ...
image = dask-gateway.813ff94f7e8c4d3da11582762e08f13c
image = dask-gateway.21911191d10b4587b83ed82da1e8006d
Get existing dask cluster: 'dask-gateway.813ff94f7e8c4d

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| lz4     | 4.4.3  | 4.4.4     | 4.4.4   |
| numpy   | 2.2.4  | 2.2.5     | 2.2.5   |
| tornado | 6.4    | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))
/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| lz4     | 4.4.3  | 4.4.4     | 4.4.4   |
| numpy   | 2.2.4  | 2.2.5     | 2.2.5   |
| tornado | 6.4    | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


In [3]:
# Create a test collection
TEST_COLLECTION_NAME = "RSPY_643_TEST_COLLECTION"
collection = create_test_collection(TEST_COLLECTION_NAME)

# Check the catalog for RSPY_643_TEST_COLLECTION
items = catalog_client.get_items(TEST_COLLECTION_NAME)
assert not list(items)

#CADIP_SESSION_FILTER = "id=S3A_20250109134406046340" # Session id "platform='sentinel-1a'" "id=S1A_20200105072204051312" S3A_20250109134406046340 | S1A_20200105072204051312
CADIP_SESSION_FILTER ="id=S1A_20200105072204051312"


09:18:54.748 [INFO] (rs_client.rs_client) Retrieving all items from collection 'agrosu:RSPY_643_TEST_COLLECTION'.


In [4]:
# Other imports
import getpass
import os
import os.path as osp
from resources import prefect_utils

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    PREFECT_BLOCK_S3.bucket_name,
    PREFECT_BLOCK_S3.bucket_folder,
    "users",
    OWNER_ID,
    "l0",
)
s3_config = osp.join(s3_base, "config")
s3_output = osp.join(s3_base, "output")

# Upload the local configuration dir to s3 bucket
await s3_upload_dir("./l0/config", s3_config)

flow_parameters = {
    "input_config_dir": s3_config,
    "payload_file": "s3/s3_l0_demo_payload_dpr_mockup_template.yaml",
    "output_data_dir": f"{s3_output}/s3",
    "owner_id": OWNER_ID,
    "collection_name": TEST_COLLECTION_NAME,
    "cadip_stac_filter": CADIP_SESSION_FILTER,
    "staging_timeout": 120,
    "use_dpr_mockup": USE_DPR_MOCKUP,
}

# Convert to json to trigger prefect flow
def to_json(my_data):
    return json.dumps(my_data).replace('"', r'\"')

09:18:55.104 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/logging_config.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/agrosu/l0/config/logging_config.yaml'.

09:18:55.107 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/l0_processor_configuration_dpr_mockup.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/agrosu/l0/config/s3/l0_processor_configuration_dpr_mockup.yaml'.

09:18:55.108 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/l0_processor_configuration_3A.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/agrosu/l0/config/s3/l0_processor_configuration_3A.yaml'.

09:18:55.110 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/s3_l0_demo_payload_dpr_mockup_template.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/agrosu/l0/config/s3/s3_l0_demo_payload_dpr_mockup_template.yaml'.

09:18:55.314 | INFO    | prefect.S3Bucket - Uploaded 4 files from 'l0/config' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/agrosu/l0/config/s3/s3_l0_demo_payload_dpr_mockup_template.yaml'

In [5]:
# Save cluster info to be read by our flow
os.environ["DASK_CLUSTER_EOPF_NAME"] = dask_cluster_eopf.name
os.environ["DASK_CLUSTER_STAGING_NAME"] = dask_cluster_staging.name
if cluster_mode:
    os.environ["DASK_GATEWAY_EOPF_ADDRESS"] = os.environ["DASK_GATEWAY_ADDRESS"]

# Setup adaptive scaling
#dask_gateway.adapt_cluster(dask_cluster.name, minimum=1, maximum=scale)

## 2. Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [6]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{OWNER_ID}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{PREFECT_BLOCK_S3.bucket_name}/{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}'")

# Upload local directory contents
await PREFECT_BLOCK_S3.put_directory(local_path = ".", to_path = s3_code_folder)

# It doesn't follow symlinks so upload them manually
await PREFECT_BLOCK_S3.put_directory(local_path = "./resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}"

Upload local source code to: 's3://rs-dev-cluster-temp/prefect-share/users/agrosu/code'


In [7]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./s3l0_demo_processor_with_dpr_service.yaml"

09:19:04.728 | ERROR   | opentelemetry.instrumentation.instrumentor - DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.1"
/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| lz4     | 4.4.3  | 4.4.4     | 4.4.4   |
| numpy   | 2.2.4  | 2.2.5     | 2.2.5   |
| tornado | 6.4    | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))
09:19:05.256 | WARNING | prefect.utilities.templating - Value for placeholder 'RSPY_HOST_USER' not found in provided values. Please ensure that the placeholder is spelled correctly and that the corresponding value is provided.
09:19:05.257 | WARNING | prefect.utilities.templating - Value for placeholder 'RSPY_LOCAL_MODE' not found in provided val

╭──────────────────────────────────────────────────────────────────────────────╮
│ Deployment 's3l0-demo-processor/sprint23-s3l0-demo-processor' successfully   │
│ created with id '83007d4e-dd73-4a68-9bdf-c0b73cda32de'.                      │
╰──────────────────────────────────────────────────────────────────────────────╯

View Deployment in UI: http://prefect-server.processing.svc.cluster.local:4200/deployments/deployment/83007d4e-dd73-4a68-9bdf-c0b73cda32de


To schedule a run for this deployment, use the following command:

        $ prefect deployment run 
's3l0-demo-processor/sprint23-s3l0-demo-processor'



In [8]:
deploy_name = "s3l0-demo-processor/sprint23-s3l0-demo-processor"
await prefect_utils.wait_for_deployment(deploy_name)

Finished deploying prefect flow: 's3l0-demo-processor/sprint23-s3l0-demo-processor'


## 3. Run Prefect flow

In [9]:
output_data_dir = flow_parameters["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(flow_parameters) # flow parameters

Remove existing zarr products from: 's3://rs-dev-cluster-temp/prefect-share/users/agrosu/l0/output/s3'


In [10]:
%%bash -s "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

Creating flow run for deployment 
's3l0-demo-processor/sprint23-s3l0-demo-processor'...
Created flow run 'quantum-dragon'.
└── UUID: cfa39fe2-5163-4d5b-b7c3-16265d8383cc
└── Parameters: {'input_config_dir': 's3://rs-dev-cluster-temp/prefect-share/users/agrosu/l0/config', 'payload_file': 's3/s3_l0_demo_payload_dpr_mockup_template.yaml', 'output_data_dir': 's3://rs-dev-cluster-temp/prefect-share/users/agrosu/l0/output/s3', 'owner_id': 'agrosu', 'collection_name': 'RSPY_643_TEST_COLLECTION', 'cadip_stac_filter': 'id=S1A_20200105072204051312', 'staging_timeout': 120, 'use_dpr_mockup': True}
└── Job Variables: {}
└── Scheduled start time: 2025-05-12 09:19:09 UTC (now)
└── URL: http://prefect-server.processing.svc.cluster.local:4200/runs/flow-run/cfa39fe2-5163-4d5b-b7c3-16265d8383cc
Watching flow run 'quantum-dragon'...


09:19:11.091 | INFO    | prefect - Flow run is in state 'Pending'
09:19:21.449 | INFO    | prefect - Flow run is in state 'Running'
09:19:40.348 | INFO    | prefect - Flow run is in state 'Completed'


Flow run finished successfully in 'Completed'.


In [11]:
print(f"Output products generated on: {output_data_dir!r}")

local_report_dir = osp.join("./l0", "reports", "s1.short")
print(f"Download reports locally: {local_report_dir!r}")
await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)
eopf_prod_ids = ["S03MWRL0__20221101T092439_6037_A307_T677", "S03OLCL0__20210629T044945_0119_A247_T219"]
for id in eopf_prod_ids:
    assert catalog_client.get_item(TEST_COLLECTION_NAME, id) 
   

Output products generated on: 's3://rs-dev-cluster-temp/prefect-share/users/agrosu/l0/output/s3'
Download reports locally: './l0/reports/s1.short'


## 6. Shutdown the dask clusters

In [12]:
shutdown = False
if shutdown:    
    # You can scale the clusters to 0 workers
    dask_gateway_eopf.scale_cluster(dask_cluster_eopf.name, 0)
    dask_gateway_staging.scale_cluster(dask_cluster_staging.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway_eopf, dask_cluster_eopf.name)
    shutdown_dask_clusters(dask_gateway_staging, dask_cluster_staging.name)

    # Close the python objects
    close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.

## For testing only: reset the cluster and run the flow locally from Python

In [13]:
from importlib import reload
debug_flow = False

In [14]:
if debug_flow:
    shutdown_dask_clusters(dask_gateway_staging, None)
    shutdown_dask_clusters(dask_gateway_eopf, None)
    init_dask_cluster_eopf(scale=2)
    init_dask_cluster_staging(scale=2)

    from resources.dask_utils import *
    os.environ["DASK_CLUSTER_EOPF_NAME"] = dask_cluster_eopf.name
    os.environ["DASK_CLUSTER_STAGING_NAME"] = dask_cluster_staging.name

In [15]:
if debug_flow:
    import s3l0_demo_processor
    reload(s3l0_demo_processor)
    results = s3l0_demo_processor.s3l0_demo_processor(**flow_parameters)
    display(results)